In [2]:
import os

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

In [7]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)


9

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

26

In [15]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [ ]:
#### agent = tools , llm , prompt


In [ ]:
@tool
def retriever_tool(query:str):
    """ 
        This tool can help you to retrieve the relevent data of the PDF documents and these PDF 
        documents have details about the medical report
    """
    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context    


In [55]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [56]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [57]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [58]:
query = "What is the name of Patient, and what is the name of Doctors"
response = agent.invoke({'messages':[{"role":"user","content":query}]})

Tool Called:  Patient and Doctor names


In [59]:
result = response["messages"][-1].content

In [60]:
print(result)

The name of the patient is Ms. Nikita Chudhary, and the names of the doctors are Dr. Nitin Nahar, Dr. Swapnil Gupta, and Dr. Kiran Bhargava Pathak.
